# Random Forest (Quantile Regression Forest) — Dự báo lưu lượng đến (Qvào)

Global model — 1 bộ 24 Quantile Regression Forest (1/horizon, direct
multi-horizon 1..24h) dùng chung cho cả 16 hồ (reservoir index là 1 feature
one-hot), quantile P10/P50/P90 — cùng contract với LSTM/XGBoost.

Dữ liệu: tabular hoá từ dataset `LSTM_Py_Backend_v2` đã build sẵn (dòng cuối
của mỗi hindcast window = "hiện tại", đã chứa lag/rolling feature nén 240h
quá khứ) — không cần Excel gốc, không dùng mưa dự báo oracle (tránh
train/serve mismatch, xem `data/tabular_dataset.py`).

**Không cần GPU** — Random Forest chạy CPU thuần.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "quantile-forest", "huggingface_hub"])


In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
import joblib
from quantile_forest import RandomForestQuantileRegressor


## config/reservoirs.py + config/settings.py

In [ ]:
# config/reservoirs.py
# Copy nguyen tu LSTM_Py_Backend_v2/config/reservoirs.py -- 16 ho Quang Nam/Da Nang.
# "idx" dung lam chi so one-hot reservoir cho model GLOBAL (RF dung chung 1 bo
# Quantile Regression Forest cho ca 16 ho, khac LSTM_Py_Backend_v2 train rieng
# tung ho -- xem README.md).

RESERVOIRS = {
    1: {"idx": 0, "name": "HO A VUONG", "lat": 15.815, "lon": 107.63, "river_basin": "Vu Gia"},
    2: {"idx": 1, "name": "HO DAK MI 4", "lat": 15.45285, "lon": 107.83250, "river_basin": "Vu Gia"},
    3: {"idx": 2, "name": "HO SONG BUNG 4", "lat": 15.726, "lon": 107.637, "river_basin": "Vu Gia"},
    4: {"idx": 3, "name": "HO SONG TRANH 2", "lat": 15.326, "lon": 108.125, "river_basin": "Thu Bồn"},
    7: {"idx": 4, "name": "HO SONG BUNG 4A", "lat": 15.765, "lon": 107.679, "river_basin": "Vu Gia"},
    8: {"idx": 5, "name": "HO SONG BUNG 5", "lat": 15.808, "lon": 107.7473, "river_basin": "Vu Gia"},
    9: {"idx": 6, "name": "HO SONG BUNG 2", "lat": 15.7145, "lon": 107.3970, "river_basin": "Vu Gia"},
    11: {"idx": 7, "name": "HO SONG BUNG 6", "lat": 15.82, "lon": 107.78, "river_basin": "Vu Gia"},
    12: {"idx": 8, "name": "HO SONG TRANH 3", "lat": 15.4445, "lon": 108.1430, "river_basin": "Thu Bồn"},
    13: {"idx": 9, "name": "HO ZA HUNG", "lat": 15.86005, "lon": 107.654, "river_basin": "Vu Gia"},
    14: {"idx": 10, "name": "HO DAK MI 3", "lat": 15.33, "lon": 107.81, "river_basin": "Vu Gia"},
    15: {"idx": 11, "name": "HO KHE DIEN", "lat": 15.71279, "lon": 107.92872, "river_basin": "Thu Bồn"},
    16: {"idx": 12, "name": "HO SONG CON 2", "lat": 15.90558, "lon": 107.8234, "river_basin": "Vu Gia"},
    17: {"idx": 13, "name": "HO SONG TRANH 4", "lat": 15.53666, "lon": 108.152, "river_basin": "Thu Bồn"},
    18: {"idx": 14, "name": "HO DAK MI 2", "lat": 15.23832, "lon": 107.8100, "river_basin": "Vu Gia"},
    19: {"idx": 15, "name": "HO DAK MI 4C", "lat": 15.4643, "lon": 107.92893, "river_basin": "Thu Bồn"},
}

NUM_RESERVOIRS = len(RESERVOIRS)


In [ ]:
# config/settings.py
HORIZON = 24                    # 24h du bao (giong LSTM/XGBoost)
QUANTILES = [0.1, 0.5, 0.9]     # P10/P50/P90 -- dung contract voi ForecastRF (Node)

# Fixed-date split -- GIONG HET LSTM_Py_Backend_v2/config/settings.py
# (ReservoirLSTMConfig.train_end/val_start/val_end/test_start) de so NSE
# cong bang giua RF/XGBoost/LSTM tren cung 1 khoang test.
TRAIN_END = "2024-08-31"
VAL_START = "2024-09-01"
VAL_END = "2025-01-01"
TEST_START = "2025-09-01"

# Nguon backup Hugging Face khi khong co data local/Kaggle input -- xem
# LSTM_Py_Backend_v2/kaggle/generate_notebook_all.py (cung 1 nguon).
HF_REPO_ID = "Anvo2004/dataset_all_lake"
HF_ZIP_FILENAME = "datasets_all_reservoirs.zip"


## data/tabular_dataset.py

In [ ]:
# data/tabular_dataset.py
"""
Doc du lieu tabular tu dataset LSTM_Py_Backend_v2 da build san (sliding-window
hindcast, xem LSTM_Py_Backend_v2/data/dataset_builder.py) -- KHONG con phu
thuoc Data_Tung_Ho_Ma_Tran_Rong/ (Excel goc, da bi xoa khoi may local).

Chien luoc: lay dong CUOI CUNG cua moi hindcast window (= "hien tai", da chua
day du lag/rolling feature nen cua 240h qua khu) lam 1 dong tabular, target la
24 buoc inflow (sqrt-space) tiep theo -- direct multi-horizon, KHONG dung
X_nwp (mua du bao oracle) lam input:

  Ly do KHONG dung X_nwp: X_nwp trong dataset build tu inflow/rain THUC TE
  cua chinh khoang thoi gian tuong lai (oracle), trong khi luc serving thuc te
  chi co du bao Open-Meteo (co sai so) -- gay train/serve mismatch. Bo X_nwp
  giup RF/XGBoost khong gap mismatch nay (doi lai la khong tan dung duoc tin
  hieu mua du bao, nhung cac dac trung rain_*h/inflow_*h_avg tich luy toi 7
  ngay qua khu da nam bat phan lon dieu kien am dat/xu huong dong chay).

Tim nguon du lieu theo thu tu uu tien:
  1. Local dev: ../LSTM_Py_Backend_v2/datasets/<Ten_Ho>/v2_*.npy
  2. Kaggle: /kaggle/input/**/v2_X_hindcast.npy (dataset da attach vao notebook)
  3. Hugging Face Hub: Anvo2004/dataset_all_lake (fallback tu dong, cung nguon
     LSTM_Py_Backend_v2/kaggle/generate_notebook_all.py dang dung)
"""
import os
import numpy as np




_LOCAL_CANDIDATES = [
    os.path.join("..", "LSTM_Py_Backend_v2", "datasets"),
    os.path.join("LSTM_Py_Backend_v2", "datasets"),
]


def _find_dataset_dirs() -> dict:
    """Tra ve {reservoir_key: folder_path chua v2_X_hindcast.npy}."""
    for local_root in _LOCAL_CANDIDATES:
        if os.path.isdir(local_root):
            found = {}
            for name in os.listdir(local_root):
                p = os.path.join(local_root, name)
                if os.path.exists(os.path.join(p, "v2_X_hindcast.npy")):
                    found[name] = p
            if found:
                print(f"[data] Dung nguon local: {local_root}/ ({len(found)} ho)")
                return found

    kaggle_root = "/kaggle/input"
    if os.path.isdir(kaggle_root):
        found = {}
        for root, _, files in os.walk(kaggle_root):
            if "v2_X_hindcast.npy" in files:
                found[os.path.basename(root)] = root
        if found:
            print(f"[data] Dung nguon Kaggle input: {kaggle_root} ({len(found)} ho)")
            return found

    print(f"[data] Khong tim thay data local/Kaggle -> tai tu Hugging Face '{HF_REPO_ID}'...")
    from huggingface_hub import hf_hub_download
    import zipfile
    zip_path = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_ZIP_FILENAME, repo_type="dataset")
    extract_dir = "/kaggle/working/datasets" if os.path.isdir("/kaggle/working") else "./_hf_datasets"
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    found = {}
    for root, _, files in os.walk(extract_dir):
        if "v2_X_hindcast.npy" in files:
            found[os.path.basename(root)] = root
    if not found:
        raise FileNotFoundError(
            "Khong tim thay v2_X_hindcast.npy o local, Kaggle input, lan Hugging Face."
        )
    print(f"[data] Da tai va giai nen tu Hugging Face ({len(found)} ho)")
    return found


def build_tabular_dataset():
    """
    Tra ve:
      X   (N, n_base_features + NUM_RESERVOIRS)  float32 -- da gom one-hot reservoir
      y   (N, HORIZON)                            float32 -- sqrt-space, da cap outlier
      rid (N,)                                    int64   -- reservoir idx (0..15)
      ts  (N,)                                     datetime64[s]
    """
    dirs = _find_dataset_dirs()
    key_to_idx = {info["name"].replace(" ", "_"): info["idx"] for _, info in RESERVOIRS.items()}

    X_list, y_list, rid_list, ts_list = [], [], [], []
    for key, path in sorted(dirs.items()):
        if key not in key_to_idx:
            print(f"  [SKIP] {key}: khong co trong config/reservoirs.py")
            continue
        idx = key_to_idx[key]
        X_hind = np.load(os.path.join(path, "v2_X_hindcast.npy"), mmap_mode="r")
        y = np.load(os.path.join(path, "v2_y.npy"))
        ts = np.load(os.path.join(path, "v2_timestamps.npy"))

        X_last = np.asarray(X_hind[:, -1, :], dtype=np.float32)  # dong cuoi = "hien tai"
        onehot = np.zeros((len(X_last), NUM_RESERVOIRS), dtype=np.float32)
        onehot[:, idx] = 1.0

        X_list.append(np.concatenate([X_last, onehot], axis=1))
        y_list.append(np.asarray(y, dtype=np.float32))
        rid_list.append(np.full(len(X_last), idx, dtype=np.int64))
        ts_list.append(ts)
        print(f"  [{key}] {len(X_last):,} samples")

    if not X_list:
        raise RuntimeError("Khong build duoc mau nao -- kiem tra lai thu muc dataset.")

    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    rid = np.concatenate(rid_list, axis=0)
    ts = np.concatenate(ts_list, axis=0)
    print(f"Total: {len(X):,} samples | X={X.shape} | y={y.shape}")
    return X, y, rid, ts


def split_by_date(ts: np.ndarray):
    """Fixed-date split -- xem config/settings.py, giong het LSTM_Py_Backend_v2
    (ReservoirLSTMConfig) va LSTM_Py_Backend/lstm_service (train_global.py) de
    so NSE cong bang giua ca 3 model tren cung 1 khoang test."""
    train_end_dt = np.datetime64(TRAIN_END, "s") + np.timedelta64(23, "h")
    val_start_dt = np.datetime64(VAL_START, "s")
    val_end_dt = np.datetime64(VAL_END, "s")
    test_start_dt = np.datetime64(TEST_START, "s")

    train_idx = np.where(ts <= train_end_dt)[0]
    val_idx = np.where((ts >= val_start_dt) & (ts < val_end_dt))[0]
    test_idx = np.where(ts >= test_start_dt)[0]
    return train_idx, val_idx, test_idx


## training/event_metrics.py

In [ ]:
# training/event_metrics.py
"""
Copy nguyen tu LSTM_Py_Backend_v2/training/event_metrics.py -- logic khong
phu thuoc model/framework, dung lai y het cho RF.
"""

import numpy as np


def nse_single(obs: np.ndarray, pred: np.ndarray) -> float:
    """NSE (Nash-Sutcliffe Efficiency) trên 1 mảng 1D đã flatten."""
    ss_res = float(np.sum((obs - pred) ** 2))
    ss_tot = float(np.sum((obs - obs.mean()) ** 2))
    if ss_tot < 1e-8:
        return float("nan")
    return 1.0 - ss_res / ss_tot


def kge_single(obs: np.ndarray, pred: np.ndarray) -> float:
    """KGE (Kling-Gupta Efficiency) tren 1 mang 1D -- bo sung cho NSE de
    tach ro loi do tuong quan (r), do lech bien thien (alpha), do lech
    trung binh (beta)."""
    obs_mean, pred_mean = obs.mean(), pred.mean()
    obs_std, pred_std = obs.std(), pred.std()
    if obs_std < 1e-8 or abs(obs_mean) < 1e-8 or pred_std < 1e-8:
        return float("nan")
    r = float(np.corrcoef(obs, pred)[0, 1])
    if np.isnan(r):
        return float("nan")
    alpha = pred_std / obs_std
    beta = pred_mean / obs_mean
    return 1.0 - float(np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2))


def r2_single(obs: np.ndarray, pred: np.ndarray) -> float:
    """Hệ số xác định R2 (Coefficient of Determination) giữa obs và pred."""
    ss_res = float(np.sum((obs - pred) ** 2))
    ss_tot = float(np.sum((obs - obs.mean()) ** 2))
    if ss_tot < 1e-8:
        return float("nan")
    return float(1.0 - ss_res / ss_tot)


def extract_lead_time_series(
    preds: np.ndarray,   # (N, T) point forecast, đơn vị gốc (m3/s)
    obs: np.ndarray,     # (N, T)
    lead_idx: int,       # 0-based: 0 = giờ thứ 1, 23 = giờ thứ 24, ...
):
    """Trích chuỗi liên tục obs/pred tại 1 lead-time cố định. Giả định test
    set sliding-window KHÔNG shuffle (đúng với cách build_tabular_dataset())."""
    return obs[:, lead_idx].copy(), preds[:, lead_idx].copy()


def nse_per_horizon(
    preds: np.ndarray,   # (N, T) point forecast (đơn vị gốc, không phải sqrt)
    obs: np.ndarray,     # (N, T)
    group_hours: int = 6,
) -> list:
    """NSE riêng cho từng nhóm lead-time (mặc định 6h/nhóm cho horizon 24h)."""
    N, T = preds.shape
    n_groups = (T + group_hours - 1) // group_hours
    results = []
    for g in range(n_groups):
        lo, hi = g * group_hours, min((g + 1) * group_hours, T)
        p = preds[:, lo:hi].reshape(-1)
        o = obs[:, lo:hi].reshape(-1)
        results.append({
            "group": g + 1,
            "hour_range": f"{lo + 1}-{hi}h",
            "nse": round(nse_single(o, p), 4),
            "n": int(p.size),
        })
    return results


def metrics_at_specific_horizons(
    preds: np.ndarray,   # (N, T) point forecast, m3/s
    obs: np.ndarray,     # (N, T)
    horizons: list = None,
) -> dict:
    """NSE, RMSE, MAE, RSE riêng cho từng mốc: 3h, 6h, 12h, 24h, 3d, 7d."""
    if horizons is None:
        horizons = [3, 6, 12, 24, 72, 168]

    horizon_labels = {3: "3h", 6: "6h", 12: "12h", 24: "24h", 72: "3d", 168: "7d"}

    results = {}
    N, T = preds.shape
    for h in horizons:
        label = horizon_labels.get(h, f"{h}h")
        idx = min(h - 1, T - 1)
        if idx >= 0:
            o_h = obs[:, idx]
            p_h = preds[:, idx]
            ss_res = float(np.sum((o_h - p_h) ** 2))
            ss_tot = float(np.sum((o_h - o_h.mean()) ** 2))
            nse_h = 1.0 - ss_res / ss_tot if ss_tot >= 1e-8 else float("nan")
            rmse_h = float(np.sqrt(np.mean((o_h - p_h) ** 2)))
            mae_h = float(np.mean(np.abs(o_h - p_h)))
            rse_h = float(ss_res / max(ss_tot, 1e-8))
            results[label] = {
                "nse": round(nse_h, 4), "rmse": round(rmse_h, 2),
                "mae": round(mae_h, 2), "rse": round(rse_h, 4),
            }
    return results


def picp(obs: np.ndarray, pred_low: np.ndarray, pred_high: np.ndarray) -> dict:
    """PICP (Prediction Interval Coverage Probability) -- ty le % thoi diem
    gia tri thuc te nam trong [pred_low, pred_high] (vd P10-P90). Ly tuong
    PICP ~ 0.80 voi P10/P90 -- khong phai 1.0."""
    obs = np.asarray(obs)
    pred_low = np.asarray(pred_low)
    pred_high = np.asarray(pred_high)
    inside = (obs >= pred_low) & (obs <= pred_high)
    width = pred_high - pred_low
    return {
        "picp": round(float(inside.mean()), 4),
        "mean_interval_width": round(float(width.mean()), 2),
    }


def detect_flood_events(
    obs: np.ndarray,
    threshold: float,
    min_separation: int = 24,
) -> list:
    """Tách các trận lũ riêng lẻ khỏi 1 chuỗi quan trắc liên tục."""
    T = len(obs)
    candidate = np.where(obs >= threshold)[0]
    if len(candidate) == 0:
        return []

    peak_indices = []
    i = 0
    while i < len(candidate):
        j = i
        while j + 1 < len(candidate) and candidate[j + 1] - candidate[j] <= min_separation:
            j += 1
        segment = candidate[i:j + 1]
        peak_indices.append(int(segment[np.argmax(obs[segment])]))
        i = j + 1

    events = []
    for peak_idx in peak_indices:
        start = peak_idx
        while start > 0 and obs[start - 1] <= obs[start]:
            start -= 1
        end = peak_idx
        while end + 1 < T and obs[end + 1] <= obs[end]:
            end += 1
        events.append({"start": start, "peak": peak_idx, "end": end})
    return events


def flood_event_diagnostics(
    obs: np.ndarray,
    pred: np.ndarray,
    threshold: float,
    min_separation: int = 24,
    peak_re_tolerance: float = 0.2,
) -> dict:
    """Chẩn đoán từng trận lũ riêng lẻ (NSE + sai số đỉnh + QA pass rate)."""
    events = detect_flood_events(obs, threshold, min_separation)
    if not events:
        return {
            "n_events": 0, "mean_event_nse": float("nan"),
            "peak_re_mean": float("nan"), "qa_pass_rate": float("nan"),
            "events": [],
        }

    details = []
    for ev in events:
        s, p, e = ev["start"], ev["peak"], ev["end"]
        o_seg = obs[s:e + 1]
        p_seg = pred[s:e + 1]
        ev_nse = nse_single(o_seg, p_seg)

        obs_peak = float(obs[p])
        pred_peak_in_window = float(p_seg.max()) if len(p_seg) else float("nan")
        re = abs(pred_peak_in_window - obs_peak) / max(obs_peak, 1e-6)

        details.append({
            "start": s, "peak": p, "end": e,
            "obs_peak": round(obs_peak, 2),
            "pred_peak": round(pred_peak_in_window, 2),
            "peak_re": round(re, 4),
            "event_nse": round(ev_nse, 4) if not np.isnan(ev_nse) else None,
            "qa_pass": bool(re < peak_re_tolerance),
        })

    valid_nse = [d["event_nse"] for d in details if d["event_nse"] is not None]
    return {
        "n_events": len(details),
        "mean_event_nse": round(float(np.mean(valid_nse)), 4) if valid_nse else float("nan"),
        "peak_re_mean": round(float(np.mean([d["peak_re"] for d in details])), 4),
        "qa_pass_rate": round(float(np.mean([d["qa_pass"] for d in details])), 4),
        "events": details,
    }


## training/train_rf.py

In [ ]:
# training/train_rf.py
"""
Huan luyen mo hinh Random Forest GLOBAL (16 ho chung, reservoir idx la 1
feature one-hot) de du bao luu luong den (Qvao), direct multi-horizon
(1..24h), 3 quantile P10/P50/P90 -- cung response contract voi LSTM/XGBoost.

Dung Quantile Regression Forest (thu vien quantile-forest, Meinshausen 2006)
thay vi RandomForestRegressor thuong: 1 RF cho ca 3 quantile cung luc (khong
can train 3 model rieng nhu pinball-loss approach cua XGBoost), vi RF khong
co objective quantile native nhu XGBoost -- QRF lay quantile thuc nghiem tu
phan phoi gia tri o cac leaf node thay vi chi lay trung binh.

Vi sao model GLOBAL (khong train rieng tung ho): LSTM_Py_Backend_v2 tung thu
bo global-model de train rieng tung ho va ket qua te hon han (vd A Vuong NSE
0.316 so voi 0.804 cua global model) -- xem [[project_scopus_paper_gaps]].
RF/XGBoost di theo huong global da duoc kiem chung tot hon.

Chay: python training/train_rf.py
Yeu cau: pip install quantile-forest
"""
import os
import sys
import time
import numpy as np
import joblib
from quantile_forest import RandomForestQuantileRegressor

if sys.stdout.encoding is not None and sys.stdout.encoding.lower() != "utf-8":
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")




ARTIFACT_DIR = "artifacts/rf"


def flood_sample_weight(y_train: np.ndarray) -> np.ndarray:
    """Oversampling lu dinh qua sample_weight (khong duplicate row) -- top 5%
    peak -> weight x2, top 1% -> weight x3. Giong train_global.py/train_xgb.py."""
    peak = y_train.max(axis=1)
    w = np.ones(len(y_train), dtype=np.float32)
    thr_95 = np.percentile(peak, 95)
    thr_99 = np.percentile(peak, 99)
    w[peak >= thr_95] = 2.0
    w[peak >= thr_99] = 3.0
    return w


def train():
    print("=" * 70)
    print("TRAIN RANDOM FOREST GLOBAL -- Quantile Regression Forest (direct multi-horizon)")
    print("=" * 70)

    X, y, rid, ts = build_tabular_dataset()
    train_idx, val_idx, test_idx = split_by_date(ts)
    print(f"Train : {len(train_idx):,}")
    print(f"Val   : {len(val_idx):,}  (khong dung de early-stop -- RF khong co early stopping "
          f"native nhu XGBoost/LSTM; giu val_idx de doi chieu/tune n_estimators thu cong)")
    print(f"Test  : {len(test_idx):,}")
    if len(train_idx) == 0:
        raise RuntimeError("Train set rong -- kiem tra lai dataset_ts trong LSTM_Py_Backend_v2/datasets/.")

    sample_weight = flood_sample_weight(y[train_idx])
    os.makedirs(ARTIFACT_DIR, exist_ok=True)

    t0 = time.time()
    for h in range(HORIZON):
        qrf = RandomForestQuantileRegressor(
            n_estimators=300,
            max_depth=14,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=42,
        )
        qrf.fit(X[train_idx], y[train_idx, h], sample_weight=sample_weight)
        joblib.dump(qrf, f"{ARTIFACT_DIR}/h{h + 1:02d}.joblib")
        print(f"  [OK] horizon h+{h + 1:02d}/{HORIZON}")

    elapsed = time.time() - t0
    print(f"\nDa train {HORIZON} Quantile Regression Forest trong {elapsed:.1f}s -> {ARTIFACT_DIR}/")
    print("Tiep theo: python training/evaluate_rf.py")


## training/evaluate_rf.py

In [ ]:
# training/evaluate_rf.py
"""
Danh gia mo hinh Random Forest tren tap test 2025 (flood season holdout, cung
khoang ngay voi LSTM/XGBoost -- xem config/settings.py) -- dung event_metrics.py
(NSE, KGE, R2, RMSE/MAE theo horizon cu the, PICP cho quantile coverage).

Xuat:
  ket_qua_danh_gia_2025_rf.xlsx    -- NSE/KGE/R2/MAE/RMSE + PICP theo tung ho
  ket_qua_nse_theo_gio_rf.xlsx     -- NSE theo tung nhom lead-time (6h/nhom)

Chay: python training/evaluate_rf.py
Yeu cau: da chay training/train_rf.py (co du 24 file .joblib trong artifacts/rf/).
"""
import sys
import numpy as np
import pandas as pd
import joblib

if sys.stdout.encoding is not None and sys.stdout.encoding.lower() != "utf-8":
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")




    nse_single, kge_single, r2_single, picp,
    nse_per_horizon, flood_event_diagnostics, extract_lead_time_series,
)




def load_models():
    models = {}
    for h in range(HORIZON):
        models[h] = joblib.load(f"{ARTIFACT_DIR}/h{h + 1:02d}.joblib")
    return models


def predict_all(models: dict, X: np.ndarray) -> np.ndarray:
    """Tra ve (N, HORIZON, n_quantiles), da sort quantile de chong crossing."""
    N = X.shape[0]
    preds = np.zeros((N, HORIZON, len(QUANTILES)), dtype=np.float32)
    for h in range(HORIZON):
        preds[:, h, :] = models[h].predict(X, quantiles=QUANTILES)
    preds = np.sort(preds, axis=2)
    return preds


def evaluate():
    X, y, rid, ts = build_tabular_dataset()
    _, _, test_idx = split_by_date(ts)
    if len(test_idx) == 0:
        raise RuntimeError("Test set rong -- kiem tra dataset co du lieu >= TEST_START khong.")

    print(f"Test samples: {len(test_idx):,}")
    models = load_models()

    preds = predict_all(models, X[test_idx])          # (N, HORIZON, 3), sqrt space
    med_idx = len(QUANTILES) // 2
    preds_med = preds[:, :, med_idx]
    preds_p10 = preds[:, :, 0]
    preds_p90 = preds[:, :, -1]

    # Inverse sqrt transform: x^2 (v2_y.npy da la sqrt-space, cap san)
    preds_raw   = np.clip(preds_med, 0, None) ** 2
    p10_raw     = np.clip(preds_p10, 0, None) ** 2
    p90_raw     = np.clip(preds_p90, 0, None) ** 2
    targets_raw = y[test_idx] ** 2
    rids_test   = rid[test_idx]

    idx_to_name = {info["idx"]: info["name"] for _, info in RESERVOIRS.items()}

    results_rows = []
    nse_dict = {}
    for r_idx in sorted(idx_to_name.keys()):
        mask = rids_test == r_idx
        if not mask.any():
            continue
        p_r, t_r = preds_raw[mask].reshape(-1), targets_raw[mask].reshape(-1)
        nse = nse_single(t_r, p_r)
        if np.isnan(nse):
            continue
        nse_dict[r_idx] = nse
        kge = kge_single(t_r, p_r)
        r2 = r2_single(t_r, p_r)
        mae = float(np.mean(np.abs(p_r - t_r)))
        rmse = float(np.sqrt(np.mean((p_r - t_r) ** 2)))
        cov = picp(targets_raw[mask].reshape(-1), p10_raw[mask].reshape(-1), p90_raw[mask].reshape(-1))
        name = idx_to_name[r_idx]
        results_rows.append({
            "Reservoir": name, "NSE": round(nse, 4), "KGE": round(kge, 4), "R2": round(r2, 4),
            "MAE (m³/s)": round(mae, 2), "RMSE (m³/s)": round(rmse, 2),
            "PICP (P10-P90)": cov["picp"], "Mean Interval Width": cov["mean_interval_width"],
        })
        print(f"  {name:.<30} NSE={nse:.3f}  KGE={kge:.3f}  R2={r2:.3f}  "
              f"MAE={mae:.2f}  RMSE={rmse:.2f}  PICP={cov['picp']:.2f}")

    avg_nse = sum(nse_dict.values()) / len(nse_dict) if nse_dict else 0.0
    results_rows.append({"Reservoir": "--- AVERAGE ---", "NSE": round(avg_nse, 4)})
    print(f"\n  NSE AVERAGE ({len(nse_dict)} ho): {avg_nse:.3f}")

    pd.DataFrame(results_rows).to_excel("ket_qua_danh_gia_2025_rf.xlsx", index=False)
    print("\nSaved: ket_qua_danh_gia_2025_rf.xlsx")

    print("\n" + "=" * 70)
    print("CHAN DOAN BO SUNG: NSE THEO LEAD-TIME & TUNG TRAN LU")
    print("=" * 70)

    horizon_rows = []
    for r_idx in sorted(nse_dict.keys()):
        name = idx_to_name[r_idx]
        mask = rids_test == r_idx
        p_r, t_r = preds_raw[mask], targets_raw[mask]

        for hrow in nse_per_horizon(p_r, t_r, group_hours=6):
            horizon_rows.append({"Reservoir": name, **hrow})

        obs_series, pred_series = extract_lead_time_series(p_r, t_r, lead_idx=HORIZON - 1)
        if len(obs_series) > 10 and obs_series.max() > 0:
            thr = float(np.percentile(obs_series, 90))
            diag = flood_event_diagnostics(obs_series, pred_series, threshold=thr)
            print(f"  {name:.<30} [lead={HORIZON}h] n_events={diag['n_events']:>3}  "
                  f"NSE_event={diag['mean_event_nse']}  peak_RE={diag['peak_re_mean']}  "
                  f"QA={diag['qa_pass_rate']}")

    pd.DataFrame(horizon_rows).to_excel("ket_qua_nse_theo_gio_rf.xlsx", index=False)
    print("\nSaved: ket_qua_nse_theo_gio_rf.xlsx")


## Chạy train + evaluate

In [ ]:
train()


In [ ]:
evaluate()


## Tải kết quả
Sau khi Run All xong: `/kaggle/working/artifacts/rf/` (24 file `.joblib`) +
`ket_qua_danh_gia_2025_rf.xlsx` + `ket_qua_nse_theo_gio_rf.xlsx` — tải cả 3
về, copy `artifacts/rf/` đè vào `RF_Py_Backend/artifacts/rf/` của project.
